# 14a · 학습 — **acm** / transfer · seed 0,1

`acm` = 같은 백본에서 **carry·BiMamba·MOSAIC 만 끈 것** = 우리 주장의 분모.  이 창: **2잡**.

---
`transfer` = **AlohaTransferCube-v0** (`lerobot/aloha_sim_transfer_cube_human`) — insertion 보다
**짧고 쉬운** task. Intro 가 *"짧은 task 는 ACT 와 대등, 길수록 우리가 앞선다"* 고 주장하므로
**짧은 쪽 데이터포인트**가 필요하다. 프로토콜은 insertion 과 동일(150k · lr 고정 · 5rep × 500ep).
출력 경로가 task 별로 갈려 insertion 결과와 **섞이지 않는다**.


## ⚠️ 이 노트북은 **A 몫**만 돌린다 (GPU 2장)

| 노트북 | seed | GPU |
|---|---|---|
| **14a (이 창)** | **[0, 1]** | **[0, 1]** |
| 14b_train_acm_transfer_seed23.ipynb | [2, 3] | [2, 3] |

두 창은 **다른 GPU** 를 쓰므로 동시에 띄워도 안 밟는다. 둘 다 끝나면 seed 4개가 모여
리포트(`09`/`10`/`18`)에서 **자동으로 합쳐진다**.
insertion 이 GPU 0-3 을 전부 쓰고 있다면 **그게 끝난 뒤에** 돌릴 것.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import importlib, common_final as cf
importlib.reload(cf)

TASK        = cf.SHORT_SIM       # 'transfer'
SEEDS, GPUS = cf.part('A')  # A → seeds [0, 1], GPU [0, 1]
TAGS        = cf.GROUP_ACM

print('task :', TASK, cf.v23.TASKS[TASK])
print('seeds:', SEEDS, '| GPU:', GPUS, '| 보이는 GPU:', cf.v23.available_gpus())
print('학습 :', TAGS, '| 잡:', len(TAGS) * len(SEEDS), '|', f'{cf.STEPS:,} step')

## 커맨드 확인 — dataset/env 가 **transfer**, GPU 가 맞는지

In [ ]:
for g, s in zip(GPUS, SEEDS):
    c = cf.make_train_cmd(TAGS[0], seed=s, task=TASK, gpu_id=g)
    print(' '.join(p for p in c.split()
                   if p.startswith(('CUDA_VISIBLE_DEVICES', '--dataset.repo_id', '--env.task', '--seed'))))

## 학습 (resume 자동)
첫 실행은 transfer 데이터셋을 한 번 먼저 받는다(prefetch). 안 그러면 잡들이 같은 HF 캐시에
동시 다운로드를 걸어 대부분 죽는다.

In [ ]:
jobs = cf.run_training(TAGS, SEEDS, task=TASK, gpus=GPUS)

## 상태

In [ ]:
cf.print_training_status(jobs)
print()
cf.print_ckpt_status(TAGS, SEEDS, TASK)